# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset using the `mlcroissant` library, following the FAIR data principles.

### Dataset Source
This dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json). All dataset entities are referenced by their `@id`, as per the Croissant specification.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Retrieve available record sets, fields, and their `@id`s.

**Note:** You can always inspect the structure of the dataset (available record sets and their fields/columns) via the Croissant schema. In `mlcroissant`, use `.metadata.record_sets` to explore the data model.

In [ ]:
# List available record sets and their fields by @id
print("Available record sets and their field @ids:")

for record_set in metadata.record_sets:
    print(f"- RecordSet @id: {record_set['@id']}")
    fields = record_set.get('fields', [])
    if not fields:
        print("    (No fields listed in schema)")
    else:
        for field in fields:
            if isinstance(field, dict):
                print(f"    - Field @id: {field.get('@id')}")
            else:
                print(f"    - Field @id: {field}")
    print()

## 3. Data Extraction

Next, load data from one or more record sets into DataFrames for analysis.

Update the record set and field `@id`s below to match those discovered in the data overview above. For this dataset, let's demonstrate loading all available record sets (if any).

In [ ]:
record_sets = [rs['@id'] for rs in metadata.record_sets]
dataframes = dict()

if not record_sets:
    print("No record sets are defined in the Croissant schema. You may need to consult the schema JSON or dataset files directly for more information.")
else:
    for record_set_id in record_sets:
        print(f"Loading records from RecordSet: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Columns: {df.columns.tolist()}")

    # Display the first available dataframe
    first_record_set_id = record_sets[0]
    display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

In this section, we demonstrate data processing steps such as filtering and normalization using field `@id`s.

You can refer to the Croissant schema or the output above to select valid record set and field IDs. If no dataframes were loaded (no record sets in metadata), this section will notify you.

In [ ]:
import numpy as np

if not dataframes:
    print("No dataframes loaded: skipping EDA section.")
else:
    record_set_id = first_record_set_id  # first available record set
    df = dataframes[record_set_id]

    # Attempt to select a numeric field by inspecting the dataframe
    numeric_col = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_col = col
            break
    if not numeric_col:
        print("No numeric fields detected for EDA.")
    else:
        print(f"Using numeric field '@id': {numeric_col}")

        threshold = 10
        filtered_df = df[df[numeric_col] > threshold].copy()
        print(f"Filtered records with {numeric_col} > {threshold}:")
        display(filtered_df.head())

        # Normalize (z-score) the selected numeric field
        filtered_df[numeric_col + "_normalized"] = (
            filtered_df[numeric_col] - filtered_df[numeric_col].mean()
        ) / filtered_df[numeric_col].std()
        print(f"Normalized '{numeric_col}' for filtered records:")
        display(filtered_df[[numeric_col, numeric_col + "_normalized"]].head())

        # Attempt grouping by another field (e.g., first non-numeric field)
        group_field = None
        for col in df.columns:
            if col != numeric_col and df[col].dtype == object:
                group_field = col
                break

        if group_field is not None:
            # Use .agg to avoid warnings with non-numeric columns
            grouped_df = filtered_df.groupby(group_field)[numeric_col].mean().reset_index()
            print(f"Grouped mean of '{numeric_col}' by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No suitable categorical group field found for grouping.")

## 5. Visualization

Visualize data distributions or relationships between fields.

Below, we generate a histogram and a boxplot for the numeric field analyzed above, if data is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes or not numeric_col:
    print("No numeric data available for visualization.")
else:
    plt.figure(figsize=(14, 6))

    plt.subplot(1,2,1)
    sns.histplot(df[numeric_col].dropna(), kde=True)
    plt.title(f"Histogram of {numeric_col}")

    plt.subplot(1,2,2)
    sns.boxplot(x=df[numeric_col].dropna())
    plt.title(f"Boxplot of {numeric_col}")

    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we have:
- Loaded the dataset metadata and (if available) record sets using the `mlcroissant` library,
- Explored the structure of the dataset via Croissant schema entities and `@id`s,
- Extracted and previewed tabular data from record sets,
- Performed initial filtering and normalization on a numeric field and visualized its distribution.

You are encouraged to further explore this dataset by referencing entities using their `@id` and consulting the Croissant JSON-LD schema for additional metadata, columns, and relationships relevant to your domain.